# TA Targeted Analysis — Processing Pipeline

Processes raw targeted-analysis exports into a censored, QC-assessed dataset,
following `TA Code Spec.md`.

**Read before editing:** `docs/superpowers/specs/2026-09-15-ta-processing-design.md`
records the places where this notebook deliberately differs from the spec, and why.
The short version is that the spec was written before it was checked against real
exports, and several of its exact strings do not appear in the data.

## Status

| Spec section | State |
|---|---|
| §1 Readfile | Implemented and verified |
| §2 RT Validation | Implemented and verified |
| §3 LOQ/ULOQ | Not yet written |
| §4–§11 | Not yet written |

Sections are built and verified one at a time. Nothing below §2 exists yet.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

# Show every compound when printing tables rather than an elided middle.
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

## Sample Type vocabulary

The exports label every row with a `Sample Type`. Later sections select rows by
comparing against these values, so they are defined once here and never typed as
literal strings further down.

Two of them differ from `TA Code Spec.md`. The spec says instrument blanks are
`"Blank"` and check standards are `"Check Standard"`; the exports actually use
`"Matrix Blank"` and `"Chk Std"`. Comparing against the spec's wording would match
no rows at all — and would do so silently, producing an empty result rather than an
error. The check at the end of §1 exists to catch exactly that, now and if the
instrument software changes its wording later.

In [ ]:
# Sample Type values, exactly as they appear in the exports.
CAL_STD = 'Cal Std'
UNKNOWN = 'Unknown'
INSTRUMENT_BLANK = 'Matrix Blank'   # TA Code Spec.md calls this 'Blank'
CHECK_STANDARD = 'Chk Std'          # TA Code Spec.md calls this 'Check Standard'

KNOWN_SAMPLE_TYPES = {CAL_STD, UNKNOWN, INSTRUMENT_BLANK, CHECK_STANDARD}

# Labels that appear in Calculated Amount in place of a number.
# N/F comes from the instrument; the other two are written by §3.
NOT_FOUND = 'N/F'
BELOW_LOQ = '<LOQ'
ABOVE_ULOQ = '>ULOQ'
CENSOR_LABELS = {NOT_FOUND, BELOW_LOQ, ABOVE_ULOQ}

# Compounds retired from the method. They still appear in older exports but are
# not part of the analysis, so they are dropped as the files are read and take no
# part in anything downstream. Add a compound here when it leaves the method.
#   diSAmPAP — removed from the method; not expected in future batches.
EXCLUDED_COMPOUNDS = {'diSAmPAP'}

## §1.1 — Data folder

**To run a different batch, change `DATA_FOLDER` in the cell below.** That one line
is the only place the folder is set.

Leave it as `None` and the notebook will ask you for the folder when you run the
cell — which is what a lab member opening this for the first time will get. Either
way the answer is remembered in `answers.yaml`, so the second pass over QC-adjusted
data does not ask again and an old run can be reproduced later.

`answers.yaml` is gitignored, because the path inside it is specific to your
machine. Setting `DATA_FOLDER` below always wins over whatever is saved there.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  SET THE DATA FOLDER HERE  —  this is the only place it is set.
#
#  Paste the folder holding this batch's CSV exports between the quotes,
#  keeping the r before the first quote:
#
#      DATA_FOLDER = r'F:\School Folder\...\26_08_04_Oyster_RawData'
#
#  Leave it as None to be asked for the folder when you run this cell.
# ══════════════════════════════════════════════════════════════════════════

DATA_FOLDER = None

# ══════════════════════════════════════════════════════════════════════════

ANSWERS_PATH = Path('answers.yaml')
answers = yaml.safe_load(ANSWERS_PATH.read_text()) if ANSWERS_PATH.exists() else {}
answers = answers or {}

if DATA_FOLDER:
    answers['data_folder'] = str(DATA_FOLDER)
    print('Using the folder set above.')
elif answers.get('data_folder'):
    print('Using the folder remembered in answers.yaml.')
else:
    # Paths pasted from Explorer often arrive wrapped in quotes.
    answers['data_folder'] = input('Folder holding the CSV exports: ').strip().strip('"\'')
    print('Saved to answers.yaml.')

ANSWERS_PATH.write_text(yaml.safe_dump(answers, sort_keys=False))

DATA_FOLDER = Path(answers['data_folder'])
if not DATA_FOLDER.is_dir():
    raise NotADirectoryError(f'Not a folder: {DATA_FOLDER}')

print(f'Data folder: {DATA_FOLDER}')

## §1.2 — Read every export into one master table

Each export file holds exactly one compound, so the master table is the
concatenation of all of them.

**Missing values.** Columns are read as text so pandas cannot pick a different
dtype per file depending on which sentinels that file happens to contain. Note that
pandas still applies its own missing-value detection while doing so: `N/A` is on its
default list and becomes `NaN` at read time, while `N/F` is not and survives as
text. That split is what we want — `N/A` in `Theoretical Amount` means the field
does not apply to that row, whereas `N/F` is an instrument result meaning the
compound was looked for and not found, which §2 and §3 must preserve.

**Stray rows.** At least one export (`NaDONA`) ends with an extra line naming no
sample and no compound, carrying two unlabelled numbers in `Total Area` and
`ISTD Area` and nothing else. It appears to be an artifact of the export rather than
a measurement. Every real result belongs to a sample, so rows with no
`Sample Raw File Name` are dropped — and reported by file rather than removed
quietly, because a rising count here would mean something changed upstream.

`source_file` is added so any row can be traced back to the file it came from.

In [ ]:
FILE_PATTERN = '*Quantitation_ByCompound*.csv'

export_files = sorted(DATA_FOLDER.glob(FILE_PATTERN))
if not export_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN} in {DATA_FOLDER}')

frames = []
stray_rows = []
for path in export_files:
    one_file = pd.read_csv(path, dtype=str)

    # A real measurement always names its sample. Anything else is an export
    # artifact, not data. Checked on the raw column before any renaming.
    before = len(one_file)
    one_file = one_file[one_file['Sample Raw File Name'].notna()]
    dropped = before - len(one_file)
    if dropped:
        stray_rows.append((path.name, dropped))

    one_file['source_file'] = path.name
    frames.append(one_file)

master = pd.concat(frames, ignore_index=True)

# Compounds no longer in the method take no part in the analysis.
retired = master['Compound Name'].isin(EXCLUDED_COMPOUNDS)

print(f'Files read: {len(export_files)}')
print(f'Rows:       {len(master) - retired.sum():,}')
print(f'Columns:    {master.shape[1]} (before selecting the ones the spec names)')

if retired.any():
    print(f'\nExcluded {retired.sum()} rows for compounds retired from the method:')
    for name in sorted(master.loc[retired, 'Compound Name'].unique()):
        print(f'  {name}')
    master = master[~retired].copy()

if stray_rows:
    print(f'\nStray rows dropped ({sum(count for _, count in stray_rows)} total):')
    for name, count in stray_rows:
        print(f'  {count} from {name}')
else:
    print('\nNo stray rows found.')

## §1.3 — Keep the columns the spec names

§1.3 lists fourteen columns to record and says the rest are ignored. The exports
carry thirty.

Two of the spec's names do not match the files. `Sample Name (Batch Ordering)` is
the `Sample Name` column — the exports have a separate `Sample Order` column, and
since the spec lists `Sample ID` separately, the parenthetical is read as describing
how `Sample Name` is numbered. `ISTD Actual RT` is spelled `ISTD Actual Rt`.

Missing columns raise rather than being skipped: a renamed column upstream should
stop the run, not quietly drop data the later sections depend on.

In [ ]:
SPEC_COLUMNS = [
    'Sample Raw File Name',
    'Sample Type',
    'Sample Name',          # spec: 'Sample Name (Batch Ordering)'
    'Sample ID',
    'Compound Name',
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Calculated Amount',
    'Peak Area',
    'ISTD Compound Name',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',       # spec: 'ISTD Actual RT'
]

absent = [name for name in SPEC_COLUMNS if name not in master.columns]
if absent:
    raise KeyError(f'Columns named in the spec are absent from the exports: {absent}')

master = master[SPEC_COLUMNS + ['source_file']].copy()

# Trim stray whitespace so comparisons against the constants above are reliable.
for column in master.columns:
    master[column] = master[column].str.strip()

print(f'Retained {len(SPEC_COLUMNS)} spec columns plus source_file.')

## §1.4 — Convert the numeric columns

Columns used in arithmetic are converted to numbers. Anything that cannot be parsed
— `N/F`, or `Peak index not specified` in the columns that carry it — becomes `NaN`.

**`Calculated Amount` is deliberately left as text.** §3 writes the labels `<LOQ`
and `>ULOQ` into this column, and §4.1.4 requires those labels be left in place
rather than recalculated. It therefore holds a mix of numbers and labels for the
rest of the pipeline, and every later section that does arithmetic on it must
exclude the labels explicitly and preserve them in its output.

The sentinels do **not** line up across columns, so no section may assume that a
row missing one value is missing the others. In this dataset 938 rows are `N/F` in
`Calculated Amount` but only 928 in `Method Apex RT` — ten rows have no calculated
amount yet a perfectly good retention time, which §2 has to decide what to do with.

In [ ]:
NUMERIC_COLUMNS = [
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Peak Area',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',
]

for column in NUMERIC_COLUMNS:
    master[column] = pd.to_numeric(master[column], errors='coerce')

print('Converted to numeric:')
for column in NUMERIC_COLUMNS:
    parsed = master[column].notna().sum()
    print(f'  {column:22s} {parsed:>7,} of {len(master):,} values parsed')

# Two kinds of compound, told apart by whether the compound has an internal
# standard of its own. A labelled standard (EIS/NIS) does not — it *is* the
# internal standard. Labelled standards are spiked at one fixed concentration
# rather than run as a calibration curve, so §3 gives them no LOQ and §9/§10
# exclude them. Derived once here and used wherever that split is needed.
has_own_istd = master.groupby('Compound Name')['ISTD Compound Name'].apply(lambda s: s.notna().any())
LABELLED_STANDARDS = set(has_own_istd[~has_own_istd].index)
TARGET_COMPOUNDS = set(has_own_istd[has_own_istd].index)

## §1 — Checks

What to look at before moving on to §2:

- every row has a `Sample Type`, and every value is one the notebook recognises —
  either failure stops the run here rather than silently matching nothing in §7,
  §9 or §10
- the compound count equals the number of export files, minus any compound retired
  from the method — one compound per file, so 78 files and one exclusion gives 77
- `N/F` rows are present and still readable as `N/F`, not turned into `NaN`
- `Calculated Amount` is still text; every other numeric column is `float64`
- the count of compounds carrying no `ISTD Compound Name` matches the number of
  labelled standards in the method — those are the EIS/NIS compounds §7 and §8 need

In [ ]:
print('Sample Type values observed')
print(master['Sample Type'].value_counts(dropna=False).to_string())

# Counted, not dropped: a row with no Sample Type belongs to no section of the
# spec, so it must stop the run rather than be quietly skipped.
missing_type = master['Sample Type'].isna().sum()
if missing_type:
    raise ValueError(
        f'{missing_type} row(s) have no Sample Type. '
        'Check the exports for stray or partial lines.'
    )

unrecognised = set(master['Sample Type']) - KNOWN_SAMPLE_TYPES
if unrecognised:
    raise ValueError(
        f'Unrecognised Sample Type value(s): {sorted(unrecognised)}. '
        'Add them to the vocabulary cell above and check which sections they belong to.'
    )
print('\nEvery row has a recognised Sample Type.')

print(f'\nExport files:  {len(export_files)}')
print(f'Compounds:     {master["Compound Name"].nunique()}')
print(f'Samples:       {master["Sample Raw File Name"].nunique()}')
print(f'Rows:          {len(master):,}')

not_found_rows = (master['Calculated Amount'] == NOT_FOUND).sum()
print(f'\nRows with Calculated Amount = {NOT_FOUND}: {not_found_rows:,}')

print(f'Target compounds:    {len(TARGET_COMPOUNDS)}')
print(f'Labelled standards:  {len(LABELLED_STANDARDS)}  (EIS/NIS, used in §7 and §8)')

print('\nColumn dtypes')
print(master.dtypes.to_string())

In [ ]:
# First few rows, for eyeballing that the columns line up with the exports.
master.head(10)

## §2 — Retention time validation

A compound should come off the column at the same time in every sample as it does
in its calibration standards. §2.1 requires every non-standard row to sit within
±0.4 minutes of the mean retention time of that compound's `Cal Std` rows. A peak
outside that window is not that compound, so the row is removed. The calibration
standards themselves define the reference, so they are not checked against it.

**`N/F` rows are removed here too, including calibration standards.** When the
instrument reports `N/F` there is no measurement — no retention time, no amount —
so the row carries nothing to validate and nothing to carry forward. A few of these
rows do have a valid retention time, but that does not rescue them: without a
calculated amount there is no result.

Removing `N/F` calibration standards has a consequence worth understanding. §3 sets
each compound's LOQ from its *lowest surviving* calibration level, so a compound
whose lowest standards were never detected gets a higher LOQ. That is the intended
behaviour — quantitation cannot be claimed at a level the instrument could not see.
The check below lists every compound this affects.

This section also creates the **compound table**, the spec's "compound list" — one
row per compound, starting with its reference retention time. Later sections add
their own columns: LOQ and ULOQ in §3, MDL or MRL in §5, spike recoveries in §6.

Removed rows are kept in `removed_rows` rather than discarded, because the QC report
in §11 has to be able to show what was dropped and why.

In [ ]:
RT_WINDOW_MIN = 0.4

# The compound table — the spec's "compound list". Later sections add columns.
compound_table = pd.DataFrame(index=sorted(master['Compound Name'].unique()))
compound_table.index.name = 'Compound Name'

# Reference retention time: the mean across each compound's calibration standards.
# N/F standards have no retention time, so they drop out of the mean on their own.
cal_std = master[master['Sample Type'] == CAL_STD]
compound_table['reference_rt'] = cal_std.groupby('Compound Name')['Method Apex RT'].mean()

no_reference = compound_table.index[compound_table['reference_rt'].isna()].tolist()
if no_reference:
    print(f'No Cal Std rows, so no reference RT, for: {no_reference}')

# How far each row sits from its own compound's reference.
reference_rt = master['Compound Name'].map(compound_table['reference_rt'])
rt_offset = (master['Method Apex RT'] - reference_rt).abs()

is_cal_std = master['Sample Type'] == CAL_STD
is_not_found = master['Calculated Amount'] == NOT_FOUND

# Two separate rules, deliberately not combined:
#   N/F means no measurement exists, so the row goes whatever its sample type —
#     including calibration standards, which is what raises the LOQ in §3 for a
#     compound whose lowest standards were never detected.
#   The RT window is checked on non-standards only, since the standards define it.
drop_rt = ~is_cal_std & ~is_not_found & (rt_offset > RT_WINDOW_MIN)
drop_row = is_not_found | drop_rt

removed_rows = master[drop_row].copy()
removed_rows['rt_offset'] = rt_offset[drop_row]
removed_rows['removed_because'] = f'RT outside +/-{RT_WINDOW_MIN} min'
removed_rows.loc[is_not_found[drop_row], 'removed_because'] = 'N/F, no measurement'

rows_before = len(master)
master = master[~drop_row].copy()

print(f'Rows before §2: {rows_before:,}')
print(removed_rows['removed_because'].value_counts().to_string())
print(f'Rows after §2:  {len(master):,}')

## §2 — Checks

What to look at before moving on to §3:

- the rows removed plus the rows kept add back up to the rows we started with
- no `N/F` values survive in `Calculated Amount`
- no surviving non-standard row sits more than 0.4 minutes from its reference
- every compound has a reference retention time
- the compounds losing the most rows are ones you would expect to be near the
  detection limit, not something surprising — a compound losing almost everything
  is worth investigating before trusting the rest of the run

In [ ]:
assert len(master) + len(removed_rows) == rows_before, 'rows went missing'
assert not (master['Calculated Amount'] == NOT_FOUND).any(), 'N/F survived §2'
assert compound_table['reference_rt'].notna().all(), 'a compound has no reference RT'

surviving_offset = (master['Method Apex RT'] - master['Compound Name'].map(compound_table['reference_rt'])).abs()
worst = surviving_offset[master['Sample Type'] != CAL_STD].max()
assert worst <= RT_WINDOW_MIN, f'a row survived at {worst:.3f} min from reference'
print(f'All checks passed. Widest surviving offset: {worst:.3f} min (limit {RT_WINDOW_MIN}).')

print('\nRemoved by reason and sample type')
print(removed_rows.groupby(['removed_because', 'Sample Type']).size().to_string())

# Calibration standards lost to N/F matter more than other losses: they set the
# LOQ and ULOQ in §3, so losing the low end of a curve raises that compound's LOQ.
lost_cal = removed_rows[removed_rows['Sample Type'] == CAL_STD]
if len(lost_cal):
    kept_cal = master[master['Sample Type'] == CAL_STD]
    print(f'\nCalibration levels lost ({len(lost_cal)} rows) — these raise LOQ in §3')
    for name, grp in lost_cal.groupby('Compound Name'):
        remaining = kept_cal.loc[kept_cal['Compound Name'] == name, 'Theoretical Amount']
        print(f'  {name:12s} lost {len(grp):2d} levels, {len(remaining):2d} remain,'
              f' lowest now {remaining.min():g}')

print('\nCompounds losing the most rows')
summary = pd.DataFrame({
    'removed': removed_rows.groupby('Compound Name').size(),
    'kept': master.groupby('Compound Name').size(),
}).fillna(0).astype(int)
print(summary.sort_values('removed', ascending=False).head(10).to_string())

## §3 — LOQ / ULOQ and censoring

A calibration curve only supports quantitation between its lowest and highest
standards. §3.1 sets each compound's **LOQ** to the lowest theoretical amount among
its calibration standards and its **ULOQ** to the highest, then censors any result
falling outside that range: below becomes `<LOQ`, above becomes `>ULOQ`.

Only standards that survived §2 count, so a level the instrument reported as `N/F`
cannot become the LOQ. This is where the five raised LOQs from §2 take effect, and
the check below shows how many extra results each one censors.

**Limits apply to target compounds only.** A labelled standard is spiked into every
sample at one fixed concentration rather than run as a curve, so its lowest and
highest calibration levels are the same number and an LOQ would be meaningless.
Those 32 compounds get no limits and are never censored; §7 and §8 use their peak
areas directly, and §9 and §10 exclude them explicitly.

**Censoring applies to unknowns and blanks, not to standards** — a departure from
§3.3's "every row in the master list", made deliberately. Censoring exists to stop
a result outside the quantifiable range being reported as a number. A standard is
not a reported result; it is the evidence establishing where that range lies.
Censoring them did two kinds of damage when tried:

- Endpoint standards scatter across their own limit by measurement noise. PFOA's
  top standard read 50002.554 against a 50000 theoretical — 100.005% recovery, an
  excellent point — and was labelled `>ULOQ`. Every one of the 18 affected
  calibration standards sat at an endpoint of its own curve.
- It destroyed §10's input. HFPO-DA's `Check Cal 5` read 166 against a theoretical
  200 — 83%, a pass under §10's 70–130% window — but HFPO-DA's LOQ had been raised
  to 200 by §2, so the value was overwritten with `<LOQ` and §10 could no longer
  assess it.

Standards keeping their numbers costs nothing: a standard that genuinely failed is
still visible, and the check below reports any outside 70–130% of its own
theoretical value.

**The label replaces the number in `Calculated Amount`**, as §3.3 and §3.4 require.
That column therefore holds a mix of numbers and labels from here on. Every later
section that does arithmetic on it — §4's EF, §5's MDL averaging, §6's spike
recovery — has to exclude the labels first and preserve them in its output.

In [ ]:
# LOQ and ULOQ apply to target compounds only, from the calibration levels that
# survived §2. Labelled standards are spiked at a single fixed concentration, so
# they have no curve and get no limits — they keep NaN here.
target_cal = master[master['Compound Name'].isin(TARGET_COMPOUNDS) & (master['Sample Type'] == CAL_STD)]
compound_table['loq'] = target_cal.groupby('Compound Name')['Theoretical Amount'].min()
compound_table['uloq'] = target_cal.groupby('Compound Name')['Theoretical Amount'].max()
compound_table['cal_levels_used'] = target_cal.groupby('Compound Name').size()

# Censoring applies to reported results, not to the standards that establish the
# range. Standards keep their numbers so §10 can compare check standards against
# cal standards, and so an endpoint standard scattering a fraction of a percent
# past its own theoretical value is not mislabelled as out of range.
CENSORED_SAMPLE_TYPES = {UNKNOWN, INSTRUMENT_BLANK}

# Calculated Amount holds text so the labels can live in it, so compare on a
# numeric copy and write the labels back into the text column. A labelled
# standard's limits are NaN, and every comparison against NaN is False, so those
# rows are never censored.
amount = pd.to_numeric(master['Calculated Amount'], errors='coerce')
loq = master['Compound Name'].map(compound_table['loq'])
uloq = master['Compound Name'].map(compound_table['uloq'])

censorable = master['Sample Type'].isin(CENSORED_SAMPLE_TYPES)
below_loq = censorable & (amount < loq)
above_uloq = censorable & (amount > uloq)

master.loc[below_loq, 'Calculated Amount'] = BELOW_LOQ
master.loc[above_uloq, 'Calculated Amount'] = ABOVE_ULOQ

print(f'Censored {below_loq.sum():,} results as {BELOW_LOQ}')
print(f'Censored {above_uloq.sum():,} results as {ABOVE_ULOQ}')
print(f'Quantifiable target results in censorable samples: {(censorable & ~below_loq & ~above_uloq & loq.notna()).sum():,}')
print(f'Standards left uncensored: {(~censorable).sum():,}')

## §3 — Checks

What to look at before moving on to §4:

- every compound has an LOQ and a ULOQ, and the LOQ is below the ULOQ
- no surviving number sits outside its compound's range — if one did, censoring
  missed it
- the cost of the raised LOQs from §2, shown per compound: how many results were
  censored only because the lower calibration levels were lost. These would have
  been reportable numbers had those standards been detected
- censoring broken down by sample type. Unknowns and blanks below LOQ are normal.
  **A calibration or check standard falling outside its own compound's range is
  not** — it means a standard did not read back at its own concentration, which is
  worth investigating before trusting that compound's results

In [ ]:
targets = compound_table.loc[sorted(TARGET_COMPOUNDS)]
assert targets['loq'].notna().all(), 'a target compound has no LOQ'
assert (targets['loq'] < targets['uloq']).all(), 'a target compound has LOQ >= ULOQ'
assert compound_table.loc[sorted(LABELLED_STANDARDS), 'loq'].isna().all(), 'a labelled standard was given an LOQ'

still_numeric = pd.to_numeric(master['Calculated Amount'], errors='coerce')
assert not (censorable & (still_numeric < loq)).any(), 'a value below LOQ escaped censoring'
assert not (censorable & (still_numeric > uloq)).any(), 'a value above ULOQ escaped censoring'
assert still_numeric[~censorable].notna().all(), 'a standard lost its number'
print(f'All checks passed. {len(targets)} target compounds have limits, '
      f'{len(LABELLED_STANDARDS)} labelled standards correctly have none.')

print('\nCensoring by sample type (target compounds only)')
target_rows = master[master['Compound Name'].isin(TARGET_COMPOUNDS)]
label = target_rows['Calculated Amount'].where(target_rows['Calculated Amount'].isin(CENSOR_LABELS), 'quantified')
print(pd.crosstab(target_rows['Sample Type'], label).to_string())

# Standards are not censored, so an out-of-range one is reported here instead —
# a standard that did not read back at its own concentration is a QC concern.
std = target_rows[~target_rows['Sample Type'].isin(CENSORED_SAMPLE_TYPES)].copy()
std['recovery_pct'] = 100 * pd.to_numeric(std['Calculated Amount'], errors='coerce') / std['Theoretical Amount']
poor = std[(std['recovery_pct'] < 70) | (std['recovery_pct'] > 130)]
print(f'\nStandards outside 70-130% of their own theoretical value: {len(poor)}')
if len(poor):
    print(poor[['Compound Name', 'Sample Type', 'Sample ID', 'Theoretical Amount', 'recovery_pct']]
          .sort_values('recovery_pct').to_string(index=False))

# What the raised LOQs from §2 actually cost, using the levels lost there.
lost_cal = removed_rows[removed_rows['Sample Type'] == CAL_STD]
if len(lost_cal):
    print('\nResults censored only because §2 removed the lower calibration levels')
    for name, was in lost_cal.groupby('Compound Name')['Theoretical Amount'].min().items():
        now = compound_table.loc[name, 'loq']
        only_because = (censorable & (master['Compound Name'] == name) & (amount >= was) & (amount < now)).sum()
        print(f'  {name:12s} LOQ {was:g} -> {now:g},  {only_because} result(s) lost')

## §3 — QC deliverable: LOQ and ULOQ per compound

The LOQ and ULOQ are reportable QC values in their own right, not just internal
thresholds, so they are printed here as a table and kept in `compound_table` for
the QC Report in §11.

`compound_table` is where every per-compound QC value accumulates as the pipeline
runs: the reference retention time from §2, the limits below, MDL or MRL from §5,
spike recoveries from §6. §11 reports it rather than recalculating anything.

`cal_levels_used` is included because an LOQ means something different when it
rests on 8 calibration levels than on 12 — it records how much curve is actually
behind each limit.

In [ ]:
loq_report = compound_table.loc[sorted(TARGET_COMPOUNDS), ['loq', 'uloq', 'cal_levels_used']]
loq_report = loq_report.rename(columns={'loq': 'LOQ', 'uloq': 'ULOQ', 'cal_levels_used': 'Cal levels'})

print(f'LOQ / ULOQ per target compound ({len(loq_report)} compounds)')
print(loq_report.sort_index().to_string())

reduced = loq_report[loq_report['Cal levels'] < loq_report['Cal levels'].max()]
if len(reduced):
    print(f'\n{len(reduced)} compound(s) rest on fewer than the full '
          f'{loq_report["Cal levels"].max()} calibration levels:')
    print(reduced.sort_values('Cal levels').to_string())